# **Sample dataset**

In [14]:
# Defining a simple dataset
attribute_names = ['age', 'income', 'student', 'credit_rate']
class_name = 'default'

data1 = {
    'age': ['youth', 'youth', 'middle_age', 'senior', 'senior', 'senior', 'middle_age', 'youth', 'youth', 'senior', 'youth', 'middle_age', 'middle_age', 'senior'],
    'income': ['high', 'high', 'high', 'medium', 'low', 'low', 'low','medium', 'medium', 'medium', 'medium', 'medium', 'high', 'medium'],
    'student': ['no', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no'],
    'credit_rate': ['fair', 'excellent', 'fair', 'fair', 'fair', 'excellent', 'excellent', 'fair', 'fair', 'fair', 'excellent', 'excellent', 'fair', 'excellent'],
    'default': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
}

df1 = pd.DataFrame(data1, columns=data1.keys())

In [15]:
#Step 1: Calculate Gini(D)
def gini_impurity(value_counts):
  n = value_counts.sum()
  p_sum = 0
  for key in value_counts.keys():
    p_sum += (value_counts[key] / n)**2
  gini = 1 - p_sum
  return gini

class_value_counts = df1[class_name].value_counts()
print(f'Number of sample in each class is:\n{class_value_counts}')

gini_class = gini_impurity(class_value_counts)
print(f'Gini(D) of the class is {gini_class:.3f}')

Number of sample in each class is:
default
yes    9
no     5
Name: count, dtype: int64
Gini(D) of the class is 0.459


In [19]:
#Step2 : Calculate the weighted Gini impurity for each candidate attribute
def gini_split_a(attribute_name):
  attribute_values = df1[attribute_name].value_counts()
  gini_A = 0
  for key in attribute_values.keys():
    df_k = df1[class_name][df1[attribute_name] == key].value_counts()
    n_k  = attribute_values[key]
    n = df1.shape[0]
    gini_A = gini_A + ((n_k/n))*gini_impurity(df_k)
  return gini_A

gini_attribute = {}
for key in attribute_names:
  gini_attribute[key] = gini_split_a(key)
  print(f'Gini({key}) = {gini_attribute[key]:.3f}')

Gini(age) = 0.343
Gini(income) = 0.442
Gini(student) = 0.367
Gini(credit_rate) = 0.429


# **Use Library**

### Model evaluation

Because fraud is the minority class, evaluate the model with the confusion matrix, precision, recall, and F1-score instead of relying only on accuracy.



In [2]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

In [4]:
# Download creditcard.csv from Google Drive

!pip install -q gdown

import gdown
import pandas as pd

file_id="1Cph-i553fx0p3apMDJwn4PY_9aVswOz7"
output_file ="creditcard.csv"

gdown.download(
    id=file_id,
    output=output_file,
    quiet=False
)

creditdata_df = pd.read_csv(output_file)

print('Dataset shape', creditdata_df.shape)
creditdata_df.head()

Downloading...
From (original): https://drive.google.com/uc?id=1Cph-i553fx0p3apMDJwn4PY_9aVswOz7
From (redirected): https://drive.google.com/uc?id=1Cph-i553fx0p3apMDJwn4PY_9aVswOz7&confirm=t&uuid=d2aa01ff-2f2c-4c5a-bf76-e99176410a61
To: /content/creditcard.csv
100%|██████████| 151M/151M [00:01<00:00, 86.6MB/s]


Dataset shape (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [20]:
X = creditdata_df.drop("Class", axis = 1)
y = creditdata_dfy = creditdata_df["Class"]


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y,
)
classifier = DecisionTreeClassifier()
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

In [27]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

confusion_mat = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(confusion_mat)


Accuracy: 0.9991
Confusion matrix:
[[56839    25]
 [   25    73]]


In [28]:
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
fscore = f1_score(y_test, y_pred, pos_label=1)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {fscore:.4f}")


Precision: 0.7449
Recall   : 0.7449
F1-score : 0.7449
